# Lesson 30 Lab — End-to-End Project: A Serviceable INT4 Plan for a 70B-Class Model

**Puzzle:** What evidence is required to move from a four-bit checkpoint to a serviceable 70B deployment plan?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

A serviceable 70B INT4 project is a sequence of gates, not a conversion command. Weight fit enables engine work; engine identity enables quality and load testing; passing quality, SLO, capacity, observability, canary, and rollback gates enables production. Any unexecuted critical gate keeps the decision at `not ready`.


## 0. Predict before running

1. Predict whether ideal 70B INT4 weights fit after reserve on the recorded RTX 5090.
2. Evaluate which deployment gates can be answered by a toy mixed-bit matrix and which require a real engine.
3. Write the minimum reversal conditions that would move the final decision toward canary.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

A serviceable 70B plan joins model revision, quantization/calibration, hardware topology, engine, cache policy, quality suite, workload/SLO, capacity/cost, observability, ownership, and rollback.

- The plan joins memory feasibility, backend compatibility, quality gates, performance SLOs, observability, and rollback.
- A 70B arithmetic ledger is not a successful model load.
- Every unsupported or unmeasured gate remains explicit rather than being filled with optimism.


## 2. Derive the mechanism

The project is a gate graph rather than one conversion command: memory feasibility enables engine build; engine evidence enables quality/performance tests; only passing all critical gates enables canary.

The gate graph begins with immutable model/recipe identity and capacity arithmetic. It then requires a supported backend build and operator trace, frozen quality suite, representative service load, cost/capacity margin, observability, owner, canary plan, and tested rollback. Dependencies matter: service SLO is undefined before a loadable engine exists.

A toy mixed-bit probe can validate the idea of fallback and a numeric threshold, but it cannot answer 70B task quality. Likewise, ideal `P/2` bytes ignores scale metadata and unquantized layers. Marking those distinctions in the final decision is part of the deliverable.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "30-end-to-end-70b-plan"
device = require_cuda()
torch.manual_seed(2026 + 30)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | BF16 rollback concept and unexecuted production gates |
| Candidate | ideal 70B INT4 capacity plus a toy mixed-bit numerical probe |
| Held constant | live GPU memory, 70B parameter count, 10% reserve, fixed toy threshold |
| Measurements | ideal weight GiB, fit boolean, toy RMSE/cosine, six gate booleans, final decision |
| Evidence | `capacity-model` |

**Experiment:** Combine live GPU capacity, a small CUDA mixed-bit quality probe, and a gate matrix to produce a bounded 70B deployment decision.


## 5. Read the experiment code

The final lab combines live capacity arithmetic and a small mixed-bit CUDA probe, then returns `not_ready_for_service` because the 70B engine, quality, and service gates were not executed.

The notebook reads live capacity, computes ideal INT4 bytes, runs a small CUDA mixed-bit matrix probe, and builds a gate dictionary. It sets engine, quality-suite, and service-SLO gates false because those experiments were not run. The final decision is derived from all gates rather than written optimistically.

This makes the notebook an executable deployment-plan skeleton. It is not a 70B load test, quantized checkpoint, or cost benchmark.

Only after these variables match the protocol should the cell be executed.


In [2]:
free,total=torch.cuda.mem_get_info(); params=70_000_000_000; ideal_int4=params*0.5; reserve=total*0.1; fit=ideal_int4 < total-reserve
w=torch.randn(1024,1024,device=device); x=torch.randn(64,1024,device=device); ref=x@w.t(); q4=symmetric_quantize(w,bits=4,group_size=128)[2]; q8=symmetric_quantize(w,bits=8,group_size=128)[2]
sensitive=torch.arange(0,1024,64,device=device); mixed=q4.clone(); mixed[:,sensitive]=q8[:,sensitive]; err=error_metrics(ref,x@mixed.t())
gates={"single_gpu_ideal_weight_fit":fit,"backend_engine_built":False,"quality_suite_passed":False,"service_slo_passed":False,
       "toy_mixed_bit_rmse_lte_2":err["rmse"]<=2.0,"rollback_artifact_defined":True}
decision="not_ready_for_service" if not all(gates.values()) else "ready_for_canary"
result=base_result(30,"capacity-model"); result.update({"live_gpu_total_gib":round(total/2**30,3),"ideal_int4_weight_gib":round(ideal_int4/2**30,3),
    "toy_mixed_bit_error":err,"deployment_gates":gates,"decision":decision,
    "conclusion":"The gate matrix kept unexecuted 70B engine, quality, and service tests explicit; arithmetic compression alone was insufficient."})


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Live GPU total | 31.358 GiB |
| Ideal INT4 weights | 32.596 GiB |
| Single-GPU ideal fit | no |
| Toy mixed-bit RMSE | 3.720175 |
| Quality suite passed | no |
| Service SLO passed | no |
| Decision | not_ready_for_service |


## 7. Interpret rather than merely print

Ideal INT4 weights were 32.596 GiB versus 31.358 GiB total GPU memory, so single-GPU weight fit failed before metadata or reserve. The toy mixed-bit probe produced RMSE 3.720175, above its threshold of 2, although cosine was 0.993368. Rollback identity was defined, but engine build, quality suite, and service SLO were all false. The derived decision was `not_ready_for_service`.

This is the correct outcome: arithmetic compression and one toy probe cannot fill missing production evidence. The gate matrix tells the next engineer exactly what remains rather than converting absence into a success claim.

**Inspection rule:** The notebook can approve further engineering or reject single-GPU feasibility; it cannot claim a 70B engine benchmark without loading one.


## 8. Keep the evidence label honest

This run is labeled **`capacity-model`**. The calculation uses live GPU information and/or a CUDA probe, but it remains a planning model until a named full engine, quality suite, and service workload execute.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "The gate matrix kept unexecuted 70B engine, quality, and service tests explicit; arithmetic compression alone was insufficient.",
  "decision": "not_ready_for_service",
  "deployment_gates": {
    "backend_engine_built": false,
    "quality_suite_passed": false,
    "rollback_artifact_defined": true,
    "service_slo_passed": false,
    "single_gpu_ideal_weight_fit": false,
    "toy_mixed_bit_rmse_lte_2": false
  },
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "capacity-model",
  "executed_at_utc": "2026-08-07T14:46:28+00:00",
  "ideal_int4_weight_gib": 32.596,
  "lesson": 30,
  "live_gpu_total_gib": 31.358,
  "schema_version": 1,
  "toy_mixed_bit_error": {
    "cosine": 0.99336779,
    "mae": 2.96422219,
    "max_abs": 16.94711685,
    "rmse": 3.72017455
  }
}
Saved: artifacts/rtx5090-result.j

## 9. Make the bounded decision

> A defensible plan exposes every gate, owner, artifact, and reversal condition before production optimization begins.

**Acceptance/rollback:** Leave every unexecuted gate visibly false. Require a real 70B load, native operator trace, frozen quality suite, service-load SLO, capacity margin, cost model, canary plan, and tested rollback before deployment.

**Failure analysis:** Calling ideal weight fit a successful load ignores the largest uncertainty. Allowing one high cosine score to override task failures also weakens the gate graph. A plan without owners, artifacts, deadlines, observability, and rollback rehearsal may be complete on paper but unusable during an incident.


## 10. Extend the evidence

Select a feasible multi-GPU or larger-memory target, build a pinned native engine, and capture layer/operator evidence. Run the frozen quality suite and representative load, fill capacity/cost margins, define monitoring and owners, then rehearse rollback. Only all-passing critical gates should change the decision to canary-ready.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
